# MSPR – Qualité des données
## Accidentologie routière en France (2019–2023)

**Groupe :** _(noms des membres)_  
**Date :** _(date de rendu)_

---

## Sommaire

1. [Contexte métier & problématique](#section-1)
2. [Chargement des données & profiling](#section-2)
3. [Identification des problèmes de qualité](#section-3)
4. [Règles de qualité avec seuils (Great Expectations)](#section-4)
5. [Traitements : correction, exclusion, enrichissement](#section-5)
6. [Monitoring qualité – tableau de bord](#section-6)
7. [Conclusion](#section-7)

---
<a id='section-1'></a>
## Section 1 – Contexte métier & problématique

### 1.1 Contexte métier

La **Sécurité Routière** (rattachée au Ministère de l'Intérieur) collecte chaque année les données de tous les accidents corporels survenus sur les routes françaises. Ces données, appelées **BAAC** (Bulletin d'Analyse des Accidents Corporels de la circulation), sont publiées en open data sur [data.gouv.fr](https://www.data.gouv.fr/fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2022/).

Pour chaque accident, 4 fichiers CSV sont produits :

| Fichier | Contenu | Clé |
|---------|---------|-----|
| `caracteristiques` | Date, heure, localisation, conditions météo | `Num_Acc` |
| `lieux` | Type de voie, tracé, état de la surface | `Num_Acc` |
| `vehicules` | Type de véhicule, manœuvre, obstacle | `Num_Acc` + `id_vehicule` |
| `usagers` | Âge, sexe, gravité des blessures, équipement | `Num_Acc` + `id_vehicule` + `id_usager` |

### 1.2 Problématique

> **"Les données d'accidentologie françaises (2019–2023) sont-elles suffisamment fiables et complètes pour identifier de manière robuste les zones géographiques et les profils d'usagers les plus à risque ?"**

### 1.3 Dimensions de qualité étudiées

| Dimension | Description |
|-----------|-------------|
| **Complétude** | Proportion de valeurs renseignées |
| **Validité** | Respect des formats et plages de valeurs |
| **Cohérence** | Homogénéité des nomenclatures entre années |
| **Unicité** | Absence de doublons |
| **Cohérence inter-fichiers** | Intégrité référentielle entre les 4 fichiers |

---
<a id='section-2'></a>
## Section 2 – Chargement des données & profiling

In [ ]:
# ── Imports généraux ──────────────────────────────────────────────────────────
import os
import requests
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)
print('Environnement prêt.')

### 2.1 Téléchargement automatique depuis data.gouv.fr

In [ ]:
# URLs des fichiers CSV – Bases annuelles accidents corporels (data.gouv.fr)
# Source : https://www.data.gouv.fr/fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2022/

DATASET_URLS = {
    2021: {
        'caracteristiques': 'https://www.data.gouv.fr/fr/datasets/r/85cfdc0c-23e4-4674-9bcd-79a970d7269b',
        'lieux':            'https://www.data.gouv.fr/fr/datasets/r/8a3b5022-0db3-4a24-8f55-8b8b71bdbc0d',
        'vehicules':        'https://www.data.gouv.fr/fr/datasets/r/0e16b756-3e85-4b41-a3f5-5a6a8ab43f87',
        'usagers':          'https://www.data.gouv.fr/fr/datasets/r/3f93cf12-5dd8-4fc2-9ed7-d6ff8a4e3e22',
    },
    2022: {
        'caracteristiques': 'https://www.data.gouv.fr/fr/datasets/r/cdc52400-47c6-4dd8-b0b2-6c2c0cc046d0',
        'lieux':            'https://www.data.gouv.fr/fr/datasets/r/0e90f0bf-3e07-4e43-bbf3-6e36f5a3c0b0',
        'vehicules':        'https://www.data.gouv.fr/fr/datasets/r/21a5df43-b9b7-416c-8bbd-bd5a42de0494',
        'usagers':          'https://www.data.gouv.fr/fr/datasets/r/6fa45ec6-b9e6-4de4-b7a4-9e6f1ed37b7c',
    },
    2023: {
        'caracteristiques': 'https://www.data.gouv.fr/fr/datasets/r/f3548ba5-f2c5-4a3e-95c2-7e0a54f5fe10',
        'lieux':            'https://www.data.gouv.fr/fr/datasets/r/5f77cf99-0f90-4bc2-a53d-5be4d0bebe99',
        'vehicules':        'https://www.data.gouv.fr/fr/datasets/r/6c3c4d42-b38e-4abd-809b-8e51e01bdac0',
        'usagers':          'https://www.data.gouv.fr/fr/datasets/r/3e5b1aba-cd46-4979-9ddb-f53dc5adc6a6',
    },
}


def download_file(url: str, dest_path: str, timeout: int = 60) -> bool:
    """Télécharge un fichier si non présent localement."""
    if os.path.exists(dest_path):
        print(f'  [SKIP] {os.path.basename(dest_path)} déjà présent')
        return True
    try:
        r = requests.get(url, timeout=timeout, allow_redirects=True)
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            f.write(r.content)
        print(f'  [OK]   {os.path.basename(dest_path)} ({len(r.content)/1024:.0f} Ko)')
        return True
    except Exception as e:
        print(f'  [ERR]  {os.path.basename(dest_path)} → {e}')
        return False


print('Téléchargement des fichiers CSV...')
for year, files in DATASET_URLS.items():
    year_dir = os.path.join(DATA_DIR, str(year))
    os.makedirs(year_dir, exist_ok=True)
    print(f'\n── Année {year}')
    for ftype, url in files.items():
        dest = os.path.join(year_dir, f'{ftype}_{year}.csv')
        download_file(url, dest)

print('\nTéléchargement terminé.')

### 2.2 Chargement et consolidation

In [ ]:
def load_csv_robust(path: str) -> pd.DataFrame:
    """Charge un CSV en testant plusieurs encodages et séparateurs."""
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        for sep in [',', ';']:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:  # au moins 2 colonnes = bon séparateur
                    return df
            except Exception:
                continue
    raise ValueError(f'Impossible de charger : {path}')


raw = {ftype: {} for ftype in ['caracteristiques', 'lieux', 'vehicules', 'usagers']}

for year in DATASET_URLS.keys():
    year_dir = os.path.join(DATA_DIR, str(year))
    for ftype in raw:
        path = os.path.join(year_dir, f'{ftype}_{year}.csv')
        if os.path.exists(path):
            df = load_csv_robust(path)
            df['annee'] = year
            raw[ftype][year] = df
            print(f'{ftype} {year} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
        else:
            print(f'[MANQUANT] {path}')

In [ ]:
# Concaténation multi-années
carac  = pd.concat(raw['caracteristiques'].values(), ignore_index=True)
lieux  = pd.concat(raw['lieux'].values(),            ignore_index=True)
veh    = pd.concat(raw['vehicules'].values(),         ignore_index=True)
usag   = pd.concat(raw['usagers'].values(),           ignore_index=True)

print('\n── DataFrames consolidés ──')
for name, df in [('caracteristiques', carac), ('lieux', lieux), ('vehicules', veh), ('usagers', usag)]:
    print(f'{name:20s} : {df.shape[0]:>8,} lignes × {df.shape[1]:>2} colonnes')

### 2.3 Aperçu du fichier caractéristiques

In [ ]:
display(carac.head(5))
print('\nTypes de colonnes :')
display(carac.dtypes.to_frame('dtype'))

### 2.4 Profiling automatique (ydata-profiling)

In [ ]:
from ydata_profiling import ProfileReport

# Profiling sur un échantillon pour limiter le temps d'exécution
sample = carac.sample(min(10_000, len(carac)), random_state=42)

profile = ProfileReport(
    sample,
    title='Profiling – Caractéristiques accidents (échantillon 10k)',
    explorative=True,
    minimal=False,
)
profile.to_file('../data/profiling_report.html')
print('Rapport sauvegardé dans ../data/profiling_report.html')
profile.to_notebook_iframe()

---
<a id='section-3'></a>
## Section 3 – Identification des problèmes de qualité

### 3.1 Complétude – Valeurs manquantes

In [ ]:
def missing_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Résumé des valeurs manquantes pour un DataFrame."""
    total   = df.isnull().sum()
    pct     = (total / len(df) * 100).round(2)
    summary = pd.DataFrame({'manquants': total, 'pct_%': pct})
    summary = summary[summary['manquants'] > 0].sort_values('pct_%', ascending=False)
    summary.index.name = f'{name} – colonne'
    return summary

dfs_info = [
    ('caracteristiques', carac),
    ('lieux',            lieux),
    ('vehicules',        veh),
    ('usagers',          usag),
]

for name, df in dfs_info:
    print(f'\n══ {name.upper()} ══')
    ms = missing_summary(df, name)
    if ms.empty:
        print('  Aucune valeur manquante.')
    else:
        display(ms)

In [ ]:
# Visualisation matricielle des valeurs manquantes (caractéristiques)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
msno.matrix(carac.sample(min(5000, len(carac)), random_state=42), ax=axes[0], sparkline=False)
axes[0].set_title('Valeurs manquantes – caracteristiques (échantillon 5k)', fontsize=11)

msno.bar(carac, ax=axes[1], color='steelblue')
axes[1].set_title('Complétude par colonne – caracteristiques', fontsize=11)
plt.tight_layout()
plt.savefig('../data/missing_values.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.2 Unicité – Doublons

In [ ]:
print('══ DOUBLONS PAR FICHIER ══\n')

# Doublons exacts (toutes colonnes)
for name, df in dfs_info:
    nb_dup = df.duplicated().sum()
    print(f'{name:20s} : {nb_dup:>5,} doublons exacts ({nb_dup/len(df)*100:.2f}%)')

print()

# Doublons sur la clé primaire Num_Acc dans caracteristiques
acc_col = [c for c in carac.columns if 'Num_Acc' in c or 'num_acc' in c.lower()]
if acc_col:
    key_col = acc_col[0]
    dup_acc = carac[carac.duplicated(subset=[key_col, 'annee'], keep=False)]
    print(f'Doublons sur ({key_col}, annee) dans caracteristiques : {len(dup_acc):,} lignes')
    if not dup_acc.empty:
        display(dup_acc.head(5))
else:
    print('Colonne Num_Acc non trouvée dans caracteristiques – vérifiez le nom.')
    print('Colonnes disponibles :', carac.columns.tolist())

### 3.3 Validité – Plages de valeurs

In [ ]:
print('══ VALIDITÉ – COLONNES NUMÉRIQUES ══\n')

# -- Heure (format HHMM, attendu entre 0 et 2359)
heure_col = [c for c in carac.columns if 'hrmn' in c.lower() or 'heure' in c.lower()]
if heure_col:
    h = pd.to_numeric(carac[heure_col[0]], errors='coerce')
    invalid_h = ((h < 0) | (h > 2359)).sum()
    print(f'Heures invalides (hors [0–2359]) : {invalid_h:,} ({invalid_h/len(carac)*100:.2f}%)')

# -- Coordonnées GPS (lat France métropolitaine ≈ 41–51°, lon ≈ -5–10°)
lat_col = [c for c in carac.columns if 'lat' in c.lower()]
lon_col = [c for c in carac.columns if 'long' in c.lower()]

if lat_col and lon_col:
    lat = pd.to_numeric(carac[lat_col[0]].astype(str).str.replace(',', '.'), errors='coerce')
    lon = pd.to_numeric(carac[lon_col[0]].astype(str).str.replace(',', '.'), errors='coerce')
    # Coordonnées brutes data.gouv (×100000)
    if lat.abs().median() > 100:
        lat = lat / 100000
        lon = lon / 100000
    invalid_gps = ((lat < 41) | (lat > 52) | (lon < -6) | (lon > 10)).sum()
    missing_gps = lat.isnull().sum()
    print(f'GPS manquants  : {missing_gps:,} ({missing_gps/len(carac)*100:.2f}%)')
    print(f'GPS hors France: {invalid_gps:,} ({invalid_gps/len(carac)*100:.2f}%)')
else:
    print('Colonnes GPS non trouvées (lat/long).')

# -- Usagers : tranche d'âge
age_col = [c for c in usag.columns if 'an_nais' in c.lower() or 'age' in c.lower()]
if age_col:
    # an_nais = année de naissance
    an = pd.to_numeric(usag[age_col[0]], errors='coerce')
    age = 2023 - an
    invalid_age = ((age < 0) | (age > 120)).sum()
    print(f'Ages aberrants  : {invalid_age:,} ({invalid_age/len(usag)*100:.2f}%)')

### 3.4 Cohérence inter-années – Nomenclatures

In [ ]:
print('══ COHÉRENCE DES COLONNES PAR ANNÉE ══\n')

for name, year_dict in raw.items():
    cols_by_year = {y: set(df.columns) for y, df in year_dict.items()}
    all_cols = set.union(*cols_by_year.values()) if cols_by_year else set()
    print(f'── {name}')
    for year, cols in cols_by_year.items():
        missing_cols = all_cols - cols
        extra_cols   = cols - all_cols  # ne peut pas arriver ici, mais par cohérence
        if missing_cols:
            print(f'  {year} – colonnes absentes : {sorted(missing_cols)}')
        else:
            print(f'  {year} – OK ({len(cols)} colonnes)')
    print()

In [ ]:
# Valeurs de la colonne 'catr' (catégorie de route) – cohérence par année
catr_col = [c for c in lieux.columns if 'catr' in c.lower()]
if catr_col:
    print('Valeurs uniques de catr par année :')
    for year, df in raw['lieux'].items():
        vals = df[catr_col[0]].dropna().unique().tolist()
        print(f'  {year}: {sorted(vals)}')
else:
    print('Colonne catr non trouvée dans lieux.')

### 3.5 Cohérence inter-fichiers – Intégrité référentielle

In [ ]:
print('══ INTÉGRITÉ RÉFÉRENTIELLE ══\n')

acc_key = [c for c in carac.columns if 'Num_Acc' in c or 'num_acc' in c.lower()]
if not acc_key:
    print('Colonne Num_Acc introuvable – vérifiez le nom exact :')
    print('  caracteristiques :', carac.columns.tolist())
else:
    key = acc_key[0]
    acc_ids = set(carac[key].dropna())

    for name, df in [('lieux', lieux), ('vehicules', veh), ('usagers', usag)]:
        fkey = [c for c in df.columns if 'Num_Acc' in c or 'num_acc' in c.lower()]
        if fkey:
            child_ids = set(df[fkey[0]].dropna())
            orphans   = child_ids - acc_ids
            missing   = acc_ids  - child_ids
            print(f'  {name:15s} – orphelins : {len(orphans):>5,}  |  accidents sans {name} : {len(missing):>5,}')
        else:
            print(f'  {name:15s} – Num_Acc introuvable')

### 3.6 Synthèse visuelle des problèmes identifiés

In [ ]:
# Récapitulatif sous forme de tableau de bord textuel
issues = pd.DataFrame([
    {'Dimension':    'Complétude',
     'Fichier':      'caracteristiques',
     'Problème':     'Colonnes GPS (lat/long) avec taux élevé de NaN',
     'Criticité':    'Haute'},
    {'Dimension':    'Complétude',
     'Fichier':      'usagers',
     'Problème':     'Colonne an_nais (année de naissance) partiellement renseignée',
     'Criticité':    'Moyenne'},
    {'Dimension':    'Validité',
     'Fichier':      'caracteristiques',
     'Problème':     'Coordonnées GPS hors bbox France métropolitaine',
     'Criticité':    'Haute'},
    {'Dimension':    'Validité',
     'Fichier':      'usagers',
     'Problème':     'Âges aberrants (< 0 ou > 120 ans)',
     'Criticité':    'Moyenne'},
    {'Dimension':    'Cohérence',
     'Fichier':      'lieux',
     'Problème':     'Recodage de la variable catr entre années',
     'Criticité':    'Haute'},
    {'Dimension':    'Cohérence',
     'Fichier':      'tous',
     'Problème':     'Colonnes renommées ou absentes selon l\'année',
     'Criticité':    'Moyenne'},
    {'Dimension':    'Unicité',
     'Fichier':      'caracteristiques',
     'Problème':     'Doublons sur Num_Acc potentiels',
     'Criticité':    'Basse'},
    {'Dimension':    'Intégrité référentielle',
     'Fichier':      'usagers / vehicules',
     'Problème':     'Usagers ou véhicules orphelins (Num_Acc absent dans carac)',
     'Criticité':    'Haute'},
])

display(issues.style
    .applymap(lambda v: 'background-color: #f8d7da' if v == 'Haute'
               else ('background-color: #fff3cd' if v == 'Moyenne'
               else 'background-color: #d4edda'), subset=['Criticité'])
    .set_caption('Catalogue des problèmes de qualité identifiés'))

---
<a id='section-4'></a>
## Section 4 – Règles de qualité avec seuils (Great Expectations)

In [ ]:
import great_expectations as gx

context = gx.get_context(mode='ephemeral')
print(f'Great Expectations version : {gx.__version__}')

### 4.1 Définition des règles sur le fichier caractéristiques

In [ ]:
# Ajout d'une source de données en mémoire
datasource = context.sources.add_pandas('accidents_source')
asset_carac = datasource.add_dataframe_asset('caracteristiques')

batch_request_carac = asset_carac.build_batch_request(dataframe=carac)

# Création d'un suite d'expectations
suite_carac = context.add_expectation_suite('suite_caracteristiques')
validator = context.get_validator(
    batch_request=batch_request_carac,
    expectation_suite=suite_carac,
)

# ── Règle 1 : Unicité de Num_Acc par année ────────────────────────────────────
acc_key = [c for c in carac.columns if 'num_acc' in c.lower() or 'Num_Acc' in c][0]
validator.expect_column_values_to_not_be_null(acc_key)
validator.expect_compound_columns_to_be_unique(column_list=[acc_key, 'annee'])

# ── Règle 2 : Complétude de hrmn > 90% ───────────────────────────────────────
heure_col = [c for c in carac.columns if 'hrmn' in c.lower() or 'heure' in c.lower()]
if heure_col:
    validator.expect_column_values_to_not_be_null(
        heure_col[0],
        mostly=0.90,
        meta={'description': 'Heure de l\'accident renseignée à 90% minimum'}
    )

# ── Règle 3 : Format hrmn valide (0–2359) ─────────────────────────────────────
if heure_col:
    validator.expect_column_values_to_be_between(
        heure_col[0], min_value=0, max_value=2359, mostly=0.99,
        meta={'description': 'Format heure HHMM valide pour 99% des lignes'}
    )

# ── Règle 4 : Colonne 'annee' présente ────────────────────────────────────────
validator.expect_column_to_exist('annee')
validator.expect_column_values_to_be_in_set('annee', [2021, 2022, 2023])

# ── Règle 5 : Département non null à 95% ─────────────────────────────────────
dep_col = [c for c in carac.columns if 'dep' in c.lower()]
if dep_col:
    validator.expect_column_values_to_not_be_null(
        dep_col[0], mostly=0.95,
        meta={'description': 'Département renseigné pour 95% des accidents'}
    )

validator.save_expectation_suite(discard_failed_expectations=False)
print('Suite d\'expectations caractéristiques créée.')

### 4.2 Règles sur le fichier usagers

In [ ]:
asset_usag = datasource.add_dataframe_asset('usagers')
batch_request_usag = asset_usag.build_batch_request(dataframe=usag)
suite_usag = context.add_expectation_suite('suite_usagers')
validator_usag = context.get_validator(
    batch_request=batch_request_usag,
    expectation_suite=suite_usag,
)

# ── Règle 1 : Gravité (grav) dans les valeurs attendues ──────────────────────
grav_col = [c for c in usag.columns if 'grav' in c.lower()]
if grav_col:
    validator_usag.expect_column_values_to_be_in_set(
        grav_col[0], value_set=[1, 2, 3, 4], mostly=0.99,
        meta={'description': 'Gravité : 1=indemne, 2=tué, 3=blessé hosp., 4=blessé léger'}
    )

# ── Règle 2 : Sexe dans {1, 2} ────────────────────────────────────────────────
sexe_col = [c for c in usag.columns if 'sexe' in c.lower()]
if sexe_col:
    validator_usag.expect_column_values_to_be_in_set(
        sexe_col[0], value_set=[1, 2], mostly=0.98,
        meta={'description': 'Sexe : 1=masculin, 2=féminin'}
    )

# ── Règle 3 : Année de naissance plausible (1900–2023) ────────────────────────
age_col = [c for c in usag.columns if 'an_nais' in c.lower()]
if age_col:
    validator_usag.expect_column_values_to_be_between(
        age_col[0], min_value=1900, max_value=2023, mostly=0.98,
        meta={'description': 'Année de naissance entre 1900 et 2023'}
    )

validator_usag.save_expectation_suite(discard_failed_expectations=False)
print('Suite d\'expectations usagers créée.')

### 4.3 Validation et résultats

In [ ]:
# Validation caracteristiques
checkpoint_carac = context.add_or_update_checkpoint(
    name='checkpoint_carac',
    validator=validator,
)
result_carac = checkpoint_carac.run()

# Validation usagers
checkpoint_usag = context.add_or_update_checkpoint(
    name='checkpoint_usag',
    validator=validator_usag,
)
result_usag = checkpoint_usag.run()

# Résumé des résultats
def gx_summary(results, suite_name: str):
    rows = []
    for run in results.run_results.values():
        for r in run['validation_result']['results']:
            rows.append({
                'Suite':        suite_name,
                'Expectation':  r['expectation_config']['expectation_type'],
                'Colonne':      r['expectation_config']['kwargs'].get('column', 'N/A'),
                'Succès':       r['success'],
                'Détail':       str(r.get('result', {}).get('unexpected_percent', ''))[:6],
            })
    return pd.DataFrame(rows)

summary_carac = gx_summary(result_carac, 'caracteristiques')
summary_usag  = gx_summary(result_usag,  'usagers')
all_results   = pd.concat([summary_carac, summary_usag], ignore_index=True)

display(all_results.style.applymap(
    lambda v: 'background-color: #d4edda' if v is True else
              ('background-color: #f8d7da' if v is False else ''),
    subset=['Succès']
).set_caption('Résultats de validation Great Expectations'))

---
<a id='section-5'></a>
## Section 5 – Traitements : correction, exclusion, enrichissement

### 5.1 Normalisation des noms de colonnes

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise les noms de colonnes (minuscules, sans espaces)."""
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('-', '_', regex=False)
    )
    return df

carac = normalize_columns(carac)
lieux = normalize_columns(lieux)
veh   = normalize_columns(veh)
usag  = normalize_columns(usag)

print('Colonnes normalisées.')
print('Colonnes carac :', carac.columns.tolist())

### 5.2 Harmonisation de la nomenclature `catr` (catégorie de route)

In [ ]:
# Mapping unifié des valeurs de catr selon la documentation officielle
# Source : description des variables BAAC
CATR_LABELS = {
    1: 'Autoroute',
    2: 'Route nationale',
    3: 'Route Départementale',
    4: 'Voie Communale',
    5: 'Hors réseau public',
    6: 'Parc de stationnement',
    7: 'Routes de Métropole Urbaine',
    9: 'Autre',
}

if 'catr' in lieux.columns:
    lieux['catr'] = pd.to_numeric(lieux['catr'], errors='coerce')
    lieux['catr_label'] = lieux['catr'].map(CATR_LABELS).fillna('Inconnu')
    print('Distribution catr :')
    display(lieux['catr_label'].value_counts())
else:
    print('Colonne catr non trouvée. Colonnes lieux :', lieux.columns.tolist())

### 5.3 Correction – Coordonnées GPS

In [ ]:
lat_col = [c for c in carac.columns if c == 'lat']
lon_col = [c for c in carac.columns if c in ['long', 'lon']]

n_before = len(carac)

if lat_col and lon_col:
    carac['lat_num'] = pd.to_numeric(
        carac[lat_col[0]].astype(str).str.replace(',', '.'), errors='coerce')
    carac['lon_num'] = pd.to_numeric(
        carac[lon_col[0]].astype(str).str.replace(',', '.'), errors='coerce')

    # Ajustement si les coordonnées sont stockées × 100000
    if carac['lat_num'].abs().median() > 100:
        carac['lat_num'] = carac['lat_num'] / 100000
        carac['lon_num'] = carac['lon_num'] / 100000

    mask_valid_gps = (
        carac['lat_num'].between(41, 52) &
        carac['lon_num'].between(-6, 10)
    )
    mask_missing_gps = carac['lat_num'].isnull()

    carac_gps_valid   = carac[mask_valid_gps].copy()
    carac_gps_missing = carac[mask_missing_gps].copy()
    carac_gps_invalid = carac[~mask_valid_gps & ~mask_missing_gps].copy()

    print(f'Total lignes            : {n_before:>8,}')
    print(f'GPS valides             : {len(carac_gps_valid):>8,} ({len(carac_gps_valid)/n_before*100:.1f}%)')
    print(f'GPS manquants (NaN)     : {len(carac_gps_missing):>8,} ({len(carac_gps_missing)/n_before*100:.1f}%)')
    print(f'GPS hors France (exclus): {len(carac_gps_invalid):>8,} ({len(carac_gps_invalid)/n_before*100:.1f}%)')
else:
    print('Colonnes lat/long non trouvées. Colonnes carac :', carac.columns.tolist())
    carac_gps_valid = carac.copy()
    print(f'Aucune exclusion GPS appliquée. Lignes conservées : {len(carac_gps_valid):,}')

### 5.4 Correction – Suppression des doublons

In [ ]:
acc_key = [c for c in carac_gps_valid.columns if 'num_acc' in c.lower()]

n_before_dup = len(carac_gps_valid)
if acc_key:
    carac_clean = carac_gps_valid.drop_duplicates(subset=[acc_key[0], 'annee'], keep='first')
else:
    carac_clean = carac_gps_valid.drop_duplicates(keep='first')

n_after_dup = len(carac_clean)
print(f'Doublons supprimés : {n_before_dup - n_after_dup:,}')
print(f'Lignes conservées  : {n_after_dup:,}')

### 5.5 Enrichissement – Nouvelles variables

In [ ]:
# ── 1. Période de la journée depuis hrmn ──────────────────────────────────────
heure_col = [c for c in carac_clean.columns if 'hrmn' in c.lower()]
if heure_col:
    heure = pd.to_numeric(carac_clean[heure_col[0]], errors='coerce')
    heure_h = (heure // 100).fillna(-1).astype(int)  # garder l'heure entière

    def periode(h):
        if h < 0: return 'Inconnu'
        if  6 <= h < 10: return 'Matin'
        if 10 <= h < 13: return 'Matinée'
        if 13 <= h < 18: return 'Après-midi'
        if 18 <= h < 22: return 'Soirée'
        return 'Nuit'

    carac_clean = carac_clean.copy()
    carac_clean['periode_journee'] = heure_h.apply(periode)
    print('Distribution période journée :')
    display(carac_clean['periode_journee'].value_counts())

# ── 2. Région depuis le département ───────────────────────────────────────────
dep_col = [c for c in carac_clean.columns if c == 'dep']
if dep_col:
    DEP_REGION = {
        '01':'Auvergne-Rhône-Alpes','02':'Hauts-de-France','03':'Auvergne-Rhône-Alpes',
        '04':'PACA','05':'PACA','06':'PACA','07':'Auvergne-Rhône-Alpes',
        '08':'Grand Est','09':'Occitanie','10':'Grand Est','11':'Occitanie',
        '12':'Occitanie','13':'PACA','14':'Normandie','15':'Auvergne-Rhône-Alpes',
        '16':'Nouvelle-Aquitaine','17':'Nouvelle-Aquitaine','18':'Centre-Val de Loire',
        '19':'Nouvelle-Aquitaine','21':'Bourgogne-Franche-Comté','22':'Bretagne',
        '23':'Nouvelle-Aquitaine','24':'Nouvelle-Aquitaine','25':'Bourgogne-Franche-Comté',
        '26':'Auvergne-Rhône-Alpes','27':'Normandie','28':'Centre-Val de Loire',
        '29':'Bretagne','30':'Occitanie','31':'Occitanie','32':'Occitanie',
        '33':'Nouvelle-Aquitaine','34':'Occitanie','35':'Bretagne','36':'Centre-Val de Loire',
        '37':'Centre-Val de Loire','38':'Auvergne-Rhône-Alpes','39':'Bourgogne-Franche-Comté',
        '40':'Nouvelle-Aquitaine','41':'Centre-Val de Loire','42':'Auvergne-Rhône-Alpes',
        '43':'Auvergne-Rhône-Alpes','44':'Pays de la Loire','45':'Centre-Val de Loire',
        '46':'Occitanie','47':'Nouvelle-Aquitaine','48':'Occitanie','49':'Pays de la Loire',
        '50':'Normandie','51':'Grand Est','52':'Grand Est','53':'Pays de la Loire',
        '54':'Grand Est','55':'Grand Est','56':'Bretagne','57':'Grand Est','58':'Bourgogne-Franche-Comté',
        '59':'Hauts-de-France','60':'Hauts-de-France','61':'Normandie','62':'Hauts-de-France',
        '63':'Auvergne-Rhône-Alpes','64':'Nouvelle-Aquitaine','65':'Occitanie',
        '66':'Occitanie','67':'Grand Est','68':'Grand Est','69':'Auvergne-Rhône-Alpes',
        '70':'Bourgogne-Franche-Comté','71':'Bourgogne-Franche-Comté','72':'Pays de la Loire',
        '73':'Auvergne-Rhône-Alpes','74':'Auvergne-Rhône-Alpes','75':'Île-de-France',
        '76':'Normandie','77':'Île-de-France','78':'Île-de-France','79':'Nouvelle-Aquitaine',
        '80':'Hauts-de-France','81':'Occitanie','82':'Occitanie','83':'PACA',
        '84':'PACA','85':'Pays de la Loire','86':'Nouvelle-Aquitaine','87':'Nouvelle-Aquitaine',
        '88':'Grand Est','89':'Bourgogne-Franche-Comté','90':'Bourgogne-Franche-Comté',
        '91':'Île-de-France','92':'Île-de-France','93':'Île-de-France','94':'Île-de-France',
        '95':'Île-de-France',
        '971':'Guadeloupe','972':'Martinique','973':'Guyane','974':'La Réunion','976':'Mayotte',
    }
    dep_str = carac_clean['dep'].astype(str).str.zfill(2)
    carac_clean['region'] = dep_str.map(DEP_REGION).fillna('Autre/DOM-TOM')
    print('\nDistribution régions :')
    display(carac_clean['region'].value_counts().head(10))

In [ ]:
# ── 3. Tranche d'âge des usagers ──────────────────────────────────────────────
an_col = [c for c in usag.columns if 'an_nais' in c.lower()]
if an_col:
    usag_clean = usag.copy()
    usag_clean['age'] = 2023 - pd.to_numeric(usag_clean[an_col[0]], errors='coerce')
    # Exclusion des âges aberrants
    usag_clean = usag_clean[(usag_clean['age'] >= 0) & (usag_clean['age'] <= 120) | usag_clean['age'].isnull()]

    bins   = [0, 17, 24, 34, 44, 54, 64, 120]
    labels = ['0-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    usag_clean['tranche_age'] = pd.cut(usag_clean['age'], bins=bins, labels=labels, right=True)
    print('Distribution tranches d\'âge :')
    display(usag_clean['tranche_age'].value_counts().sort_index())
else:
    usag_clean = usag.copy()
    print('Colonne an_nais non trouvée. Colonnes usag :', usag.columns.tolist())

In [ ]:
# Sauvegarde des données traitées
carac_clean.to_csv('../data/caracteristiques_clean.csv', index=False)
usag_clean.to_csv('../data/usagers_clean.csv',          index=False)
lieux.to_csv('../data/lieux_clean.csv',                 index=False)
veh.to_csv('../data/vehicules_clean.csv',               index=False)

print('Données traitées sauvegardées dans ../data/')

---
<a id='section-6'></a>
## Section 6 – Monitoring qualité – Tableau de bord

### 6.1 Calcul des indicateurs qualité par année

In [ ]:
def compute_quality_metrics(year: int, df_carac: pd.DataFrame, df_usag: pd.DataFrame) -> dict:
    """Calcule les métriques qualité pour une année donnée."""
    sub_c = df_carac[df_carac['annee'] == year]
    sub_u = df_usag[df_usag['annee'] == year]

    n_c = len(sub_c)
    n_u = len(sub_u)

    metrics = {'annee': year, 'nb_accidents': n_c, 'nb_usagers': n_u}

    # ── Complétude ────────────────────────────────────────────────────────────
    if n_c > 0:
        metrics['completude_carac_pct'] = round((1 - sub_c.isnull().mean().mean()) * 100, 2)
        dep_col = [c for c in sub_c.columns if c == 'dep']
        if dep_col:
            metrics['completude_dep_pct'] = round(sub_c['dep'].notna().mean() * 100, 2)
        heure_col = [c for c in sub_c.columns if 'hrmn' in c.lower()]
        if heure_col:
            metrics['completude_heure_pct'] = round(sub_c[heure_col[0]].notna().mean() * 100, 2)

    # ── Validité GPS ──────────────────────────────────────────────────────────
    if 'lat_num' in sub_c.columns and n_c > 0:
        valid_gps = sub_c['lat_num'].between(41, 52).sum()
        metrics['validite_gps_pct'] = round(valid_gps / n_c * 100, 2)
    else:
        metrics['validite_gps_pct'] = None

    # ── Unicité Num_Acc ────────────────────────────────────────────────────────
    acc_key = [c for c in sub_c.columns if 'num_acc' in c.lower()]
    if acc_key and n_c > 0:
        metrics['unicite_num_acc_pct'] = round(
            (1 - sub_c.duplicated(subset=acc_key, keep=False).mean()) * 100, 2
        )

    # ── Score global ──────────────────────────────────────────────────────────
    scores = [v for k, v in metrics.items() if k.endswith('_pct') and v is not None]
    metrics['score_global_pct'] = round(np.mean(scores), 2) if scores else None

    return metrics


quality_rows = []
for year in sorted(carac_clean['annee'].unique()):
    row = compute_quality_metrics(year, carac_clean, usag_clean)
    quality_rows.append(row)

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv('../data/quality_report.csv', index=False)

display(quality_df)

### 6.2 Visualisation – Évolution des indicateurs

In [ ]:
pct_cols = [c for c in quality_df.columns if c.endswith('_pct')]

if pct_cols and len(quality_df) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Score global par année', 'Indicateurs qualité par année'),
    )

    # Score global
    if 'score_global_pct' in quality_df.columns:
        fig.add_trace(
            go.Bar(x=quality_df['annee'].astype(str),
                   y=quality_df['score_global_pct'],
                   name='Score global',
                   marker_color='steelblue'),
            row=1, col=1
        )

    # Indicateurs individuels
    colors = px.colors.qualitative.Set2
    for i, col in enumerate([c for c in pct_cols if c != 'score_global_pct']):
        fig.add_trace(
            go.Scatter(x=quality_df['annee'].astype(str),
                       y=quality_df[col],
                       name=col.replace('_pct', '').replace('_', ' ').title(),
                       mode='lines+markers',
                       line=dict(color=colors[i % len(colors)])),
            row=1, col=2
        )

    fig.update_yaxes(range=[80, 101], title_text='Score (%)')
    fig.update_layout(
        title='Tableau de bord qualité – Accidents routiers 2021–2023',
        height=450,
        legend=dict(orientation='h', yanchor='bottom', y=-0.3)
    )
    fig.show()
else:
    print('Données insuffisantes pour afficher le graphique.')

### 6.3 Analyse exploratoire post-traitement

In [ ]:
# Accidents par région (top 10)
if 'region' in carac_clean.columns:
    top_regions = carac_clean['region'].value_counts().head(10)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    top_regions.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title('Top 10 régions – nombre d\'accidents', fontsize=12)
    axes[0].set_xlabel('Nombre d\'accidents')
    axes[0].invert_yaxis()

    if 'periode_journee' in carac_clean.columns:
        ordre_periodes = ['Nuit', 'Matin', 'Matinée', 'Après-midi', 'Soirée', 'Inconnu']
        periode_counts = carac_clean['periode_journee'].value_counts().reindex(ordre_periodes, fill_value=0)
        periode_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white', rot=30)
        axes[1].set_title('Accidents par période de la journée', fontsize=12)
        axes[1].set_xlabel('')
        axes[1].set_ylabel('Nombre d\'accidents')

    plt.tight_layout()
    plt.savefig('../data/accidents_analyse.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Colonne region non disponible.')

In [ ]:
# Gravité des blessures par tranche d'âge
grav_col = [c for c in usag_clean.columns if 'grav' in c.lower()]

if grav_col and 'tranche_age' in usag_clean.columns:
    GRAV_LABELS = {1: 'Indemne', 2: 'Tué', 3: 'Blessé hospitalisé', 4: 'Blessé léger'}
    usag_clean['grav_label'] = pd.to_numeric(usag_clean[grav_col[0]], errors='coerce').map(GRAV_LABELS)

    pivot = (
        usag_clean
        .dropna(subset=['tranche_age', 'grav_label'])
        .groupby(['tranche_age', 'grav_label'])
        .size()
        .unstack(fill_value=0)
    )
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    pivot_pct.plot(
        kind='bar', stacked=True, figsize=(12, 5),
        colormap='RdYlGn_r', edgecolor='white'
    )
    plt.title('Répartition de la gravité par tranche d\'âge (%)', fontsize=12)
    plt.xlabel('Tranche d\'âge')
    plt.ylabel('% d\'usagers')
    plt.xticks(rotation=30)
    plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
    plt.tight_layout()
    plt.savefig('../data/gravite_age.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Colonnes grav ou tranche_age non disponibles.')

---
<a id='section-7'></a>
## Section 7 – Conclusion

In [ ]:
# Résumé chiffré de la démarche
print('═' * 60)
print('SYNTHÈSE DE LA DÉMARCHE QUALITÉ')
print('═' * 60)

print(f"""
Dataset   : Accidents corporels de la route – data.gouv.fr
Période   : 2021–2023
Fichiers  : 4 fichiers/an (carac., lieux, véhicules, usagers)

── Volumétrie brute ───────────────────────────────────
  Accidents   : {len(carac):>10,} lignes
  Usagers     : {len(usag):>10,} lignes
  Véhicules   : {len(veh):>10,} lignes
  Lieux       : {len(lieux):>10,} lignes

── Volumétrie après traitement ───────────────────────
  Accidents (clean) : {len(carac_clean):>10,} lignes
  Usagers (clean)   : {len(usag_clean):>10,} lignes

── Traitements appliqués ─────────────────────────────
  ✓ Normalisation des noms de colonnes
  ✓ Harmonisation nomenclature catr (catégorie route)
  ✓ Exclusion coordonnées GPS hors France
  ✓ Suppression des doublons sur Num_Acc
  ✓ Enrichissement : période_journee, région, tranche_age
  ✓ Exclusion usagers avec âge aberrant
""")

if len(quality_df) > 0 and 'score_global_pct' in quality_df.columns:
    score_moyen = quality_df['score_global_pct'].mean()
    print(f'── Score qualité moyen : {score_moyen:.1f}%')

### 7.1 Réponse à la problématique

> **Problématique :** *"Les données d'accidentologie françaises (2021–2023) sont-elles suffisamment fiables et complètes pour identifier de manière robuste les zones géographiques et les profils d'usagers les plus à risque ?"*

**Réponse nuancée :**

**Partiellement oui**, à condition d'appliquer les traitements documentés dans ce notebook.

**Points positifs :**
- La structure des données (4 fichiers liés par `Num_Acc`) est robuste et cohérente
- Les variables essentielles (gravité, département, type de route) sont bien renseignées
- Les données permettent d'identifier des tendances significatives par région et tranche d'âge

**Limites identifiées :**
- **GPS** : taux de données manquantes ou invalides élevé → analyse géospatiale fine difficile
- **Cohérence inter-années** : changements de nomenclature nécessitent une harmonisation manuelle
- **Déclaratif** : les données dépendent des déclarations des forces de l'ordre → potentiel biais de sous-déclaration pour les blessés légers
- **DOM-TOM** : données hétérogènes selon les territoires d'outre-mer

**Recommandations pour usage en production :**
1. Automatiser l'harmonisation des nomenclatures lors de chaque nouvelle livraison annuelle
2. Mettre en place les règles Great Expectations comme tests de régression sur chaque millésime
3. Documenter les variables GPS comme "indicatrices" (non fiables pour l'analyse précise)
4. Enrichir avec des données OpenStreetMap pour pallier les GPS manquants via géocodage du département/commune

In [ ]:
print('Notebook exécuté avec succès.')
print('Fichiers produits dans ../data/ :')
for f in sorted(os.listdir('../data')):
    if not f.startswith('.'):
        fpath = os.path.join('../data', f)
        if os.path.isfile(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f'  {f:40s} {size_kb:>8.1f} Ko')